In [7]:
from datetime import datetime, timedelta
from pathlib import Path
import polars as pl
import os
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
import time
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
data_path_raw = (
    Path.cwd()
    / "data/external"
    / "bars_data_.parquet"
)
data_path_raw

WindowsPath('c:/Users/zhatz/Documents/GitHub/quant-alpha-model/data/external/bars_data_.parquet')

In [9]:
data_path_raw.parent

WindowsPath('c:/Users/zhatz/Documents/GitHub/quant-alpha-model/data/external')

In [6]:
start_date = datetime(2019, 1, 6)
end_date = datetime(2025, 12, 30)

stocks = [
    "SPY",
    "NVDA",
    "AAPL",
    "MSFT",
    "AMZN",
    "GOOGL",
    "AVGO",
    "GOOG",
    "META",
    "TSLA",
    "BRK.B",
    "JPM",
    "LLY",
    "V",
    "XOM",
    "JNJ",
    "WMT",
    "MA",
    "PLTR",
    "ABBV",
    "NFLX",
    "COST",
    "BAC",
    "AMD",
    "HD",
    "PG",
]

api_key = os.getenv("APCA-API-KEY-ID")
secret_key = os.getenv("APCA-API-SECRET-KEY")

client = StockHistoricalDataClient(
    api_key=api_key,
    secret_key=secret_key,
)

# Download in 3-month chunks
chunk_size = timedelta(days=90)
current_start = start_date
all_data = []

while current_start < end_date:
    current_end = min(current_start + chunk_size, end_date)

    print(f"Fetching data from {current_start.date()} to {current_end.date()}...")

    try:
        request_params = StockBarsRequest(
            symbol_or_symbols=stocks,
            timeframe=TimeFrame.Minute,
            start=current_start,
            end=current_end,
            adjustment="all",
        )

        # Get pandas df from Alpaca
        bars_pd = client.get_stock_bars(request_params).df

        # Convert to Polars immediately
        bars_pl = pl.from_pandas(bars_pd, include_index=True)
        all_data.append(bars_pl)

        print(f"Successfully fetched {len(bars_pl)} rows")

        # Be nice to the API
        time.sleep(1)

    except Exception as e:
        print(f"Error fetching chunk: {e}")
        # Save what you have so far
        if all_data:
            pl.concat(all_data).write_parquet(f"partial_data_{current_start.date()}.parquet")
        raise

    current_start = current_end

# Combine all chunks
print("Combining all data...")
combined_df = pl.concat(all_data)

# Sort by symbol first, then timestamp
# This creates better data locality for queries
print("Sorting data...")
combined_df = combined_df.sort(["timestamp", "symbol"])

# Save with optimal settings
data_path_raw = (
    Path.cwd()
    / "data/external"
    / f"bars_data_{start_date.strftime('%Y%m%d')}_to_{end_date.strftime('%Y%m%d')}__{datetime.now().strftime('%Y%m%d')}.parquet"
)

print("Writing to parquet...")
combined_df.write_parquet(
    data_path_raw,
    compression="zstd",  # Good balance of speed and compression
    statistics=True,     # Enable statistics for better query pruning
)
print(f"Saved {len(combined_df)} total rows to {data_path_raw}")

Fetching data from 2019-01-06 to 2019-04-06...
Successfully fetched 762920 rows
Fetching data from 2019-04-06 to 2019-07-05...
Successfully fetched 733207 rows
Fetching data from 2019-07-05 to 2019-10-03...
Successfully fetched 756863 rows
Fetching data from 2019-10-03 to 2020-01-01...
Successfully fetched 726216 rows
Fetching data from 2020-01-01 to 2020-03-31...
Successfully fetched 821983 rows
Fetching data from 2020-03-31 to 2020-06-29...
Successfully fetched 839698 rows
Fetching data from 2020-06-29 to 2020-09-27...
Successfully fetched 830354 rows
Fetching data from 2020-09-27 to 2020-12-26...
Successfully fetched 867404 rows
Fetching data from 2020-12-26 to 2021-03-26...
Successfully fetched 854297 rows
Fetching data from 2021-03-26 to 2021-06-24...
Successfully fetched 816271 rows
Fetching data from 2021-06-24 to 2021-09-22...
Successfully fetched 807025 rows
Fetching data from 2021-09-22 to 2021-12-21...
Successfully fetched 849552 rows
Fetching data from 2021-12-21 to 2022-03